# UrbanSound8K — v2 CUDA (Local GPU)

**Multi-session, checkpoint-compatible with v1 and v3.**
Completed folds are skipped automatically on resume.

**Session workflow:** Re-run all cells top to bottom. Cells 0–7 take ~30 sec. Then Cell 8 (SVM, skips if done) and Cell 9 (CNN, resumes from last checkpoint).

**Cross-version:** `checkpoints/` and `progress.json` format is identical across v1/v2/v3. Copy the folder between machines to resume anywhere.

In [1]:
# ── Cell 0: Version check ────────────────────────────────────────
# Checks if local repo is in sync with GitHub before running.
import subprocess, sys, os

NOTEBOOK_VERSION = '1.0.0'  # bump this when making breaking changes to shared logic

result = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True)
print(f'Commit: {result.stdout.strip()}')

dirty = subprocess.run(['git', 'status', '--short'], capture_output=True, text=True).stdout.strip()
if dirty:
    print(f'[WARN] Uncommitted changes:\n{dirty}')
    print('       Push before switching to v3/Colab to avoid conflicts.')
else:
    print('Working tree clean.')

subprocess.run(['git', 'fetch', 'origin', '--quiet'])
behind = subprocess.run(['git', 'rev-list', 'HEAD..origin/main', '--count'],
                         capture_output=True, text=True).stdout.strip()
if behind and int(behind) > 0:
    print(f'[WARN] {behind} commit(s) behind origin/main — run: git pull')
else:
    print('Up to date with origin/main.')

print(f'Notebook version: {NOTEBOOK_VERSION}')

Commit: 
Working tree clean.
Up to date with origin/main.
Notebook version: 1.0.0


fatal: not a git repository (or any parent up to mount point /home)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [2]:
# ── Cell 1: Check hardware ───────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = torch.device('cuda')
else:
    print('[WARN] No CUDA GPU — falling back to CPU. Training will be slow.')
    DEVICE = torch.device('cpu')

print(f'Device:  {DEVICE}')
print(f'PyTorch: {torch.__version__}')

GPU:   NVIDIA GeForce GTX 1080 Ti
VRAM:  11.7 GB
Device:  cuda
PyTorch: 2.2.1+cu121


In [5]:
# ── Cell 2: Paths ────────────────────────────────────────────────
# Download dataset
import os
import shutil
import kagglehub
import json



DATASET_ROOT   = './UrbanSound8K'   # ← edit to your local path
CHECKPOINT_DIR = './checkpoints'
RESULTS_DIR    = './results'
CACHE_DIR      = './cache'

dst = DATASET_ROOT
csv_path = os.path.join(dst, 'metadata', 'UrbanSound8K.csv')

if os.path.exists(csv_path):
    print('Dataset already in Drive with correct structure. Skipping.')
else:
    # Set Kaggle credentials
    os.makedirs('./.kaggle', exist_ok=True)
    with open('./.kaggle/kaggle.json', 'w') as f:
        json.dump({
            "username":'duynguyen520',
            "key":  'KGAT_d94a621c4ded256664679dbf377f8ec4'
        }, f)
    os.chmod('./.kaggle/kaggle.json', 0o600)

    # Download
    print('Downloading UrbanSound8K...')
    src = kagglehub.dataset_download('chrisfilo/urbansound8k')
    print(f'Downloaded to: {src}')

    # Delete any previous bad copy
    if os.path.exists(dst):
        print('Removing previous copy...')
        shutil.rmtree(dst)

    # Rebuild correct structure
    os.makedirs(os.path.join(dst, 'audio'), exist_ok=True)
    os.makedirs(os.path.join(dst, 'metadata'), exist_ok=True)

    for item in os.listdir(src):
        full = os.path.join(src, item)
        if item.startswith('fold'):
            print(f'  Copying {item}...')
            shutil.copytree(full, os.path.join(dst, 'audio', item))
        elif item.endswith('.csv'):
            print(f'  Copying {item}...')
            shutil.copy2(full, os.path.join(dst, 'metadata', 'UrbanSound8K.csv'))

    # Verify
    print(f'\nmetadata/UrbanSound8K.csv: {os.path.exists(csv_path)}')
    print(f'audio/ folders: {sorted(os.listdir(os.path.join(dst, "audio")))}')
    print('Done.')

for d in [CHECKPOINT_DIR, RESULTS_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Dataset: {os.path.abspath(DATASET_ROOT)}')
print(f'Exists:  {os.path.exists(DATASET_ROOT)}')
print(f'CSV:     {os.path.exists(os.path.join(DATASET_ROOT, "metadata", "UrbanSound8K.csv"))}')

NameError: name 'userdata' is not defined

In [13]:
# ── Cell 3: Install dependencies (first time only) ───────────────
# For CUDA torch: pip install torch --index-url https://download.pytorch.org/whl/cu118
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'librosa', 'scikit-learn', 'tqdm', 'seaborn', 'matplotlib'], check=True)
print('Dependencies ready.')

Dependencies ready.


In [14]:
# ── Cell 4: Core utilities ───────────────────────────────────────
# !! Keep in sync with v1 Cell 4 and v3 Cell 5 — identical code !!
import numpy as np
import pandas as pd
import librosa
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json

CLASSES       = ['air_conditioner','car_horn','children_playing','dog_bark',
                  'drilling','engine_idling','gun_shot','jackhammer','siren','street_music']
SAMPLE_RATE   = 22050
CLIP_DURATION = 4.0
N_MFCC        = 40
N_MELS        = 128
HOP_LENGTH    = 512
N_FFT         = 2048
MEL_LENGTH    = 128

def load_audio(path, sr=SAMPLE_RATE, duration=CLIP_DURATION):
    try:
        y, _ = librosa.load(path, sr=sr, duration=duration, mono=True)
    except:
        return np.zeros(int(sr * duration), dtype=np.float32)
    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)

def extract_mfcc(y, sr=SAMPLE_RATE):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])

def extract_mel(y, sr=SAMPLE_RATE, fixed_length=MEL_LENGTH):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                          n_fft=N_FFT, hop_length=HOP_LENGTH, fmax=8000)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    if log_mel.shape[1] < fixed_length:
        log_mel = np.pad(log_mel, ((0,0),(0, fixed_length - log_mel.shape[1])),
                         constant_values=log_mel.min())
    else:
        log_mel = log_mel[:, :fixed_length]
    lo, hi = log_mel.min(), log_mel.max()
    if hi - lo > 1e-6:
        log_mel = (log_mel - lo) / (hi - lo)
    return log_mel.astype(np.float32)

def load_metadata(root):
    return pd.read_csv(os.path.join(root, 'metadata', 'UrbanSound8K.csv'))

def audio_path(root, fold, fname):
    return os.path.join(root, 'audio', f'fold{fold}', fname)

def get_fold_splits(meta):
    for fold in sorted(meta['fold'].unique()):
        yield fold, meta[meta['fold'] != fold].copy(), meta[meta['fold'] == fold].copy()

print('Core utilities loaded.')

Core utilities loaded.


In [15]:
# ── Cell 5: Dataset + Models ─────────────────────────────────────
# !! Keep in sync with v1 Cell 5 and v3 Cell 6 — identical code !!

class MelDataset(Dataset):
    def __init__(self, df, root, augment=False):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = load_audio(audio_path(self.root, row['fold'], row['slice_file_name']))
        if self.augment:
            shift = int(np.random.uniform(-0.1, 0.1) * len(y))
            y = np.roll(y, shift)
            y += np.random.normal(0, 0.005, y.shape).astype(y.dtype)
        mel = torch.from_numpy(extract_mel(y)).unsqueeze(0)
        return mel, torch.tensor(int(row['classID']), dtype=torch.long)


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.MaxPool2d(2) if pool else nn.Identity()
        )
    def forward(self, x): return self.block(x)


class UrbanCNN(nn.Module):
    def __init__(self, n_classes=10, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1,   32),
            ConvBlock(32,  64),
            ConvBlock(64,  128),
            ConvBlock(128, 256),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*8*8, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, n_classes)
        )
    def forward(self, x): return self.head(self.features(x))


print('Dataset and model classes defined.')

Dataset and model classes defined.


In [16]:
# ── Cell 6: Config ───────────────────────────────────────────────
# GPU presets:
#   RTX 3080/3090 (10-24GB): BATCH_SIZE=128, NUM_WORKERS=4
#   RTX 4090 (24GB):         BATCH_SIZE=256, NUM_WORKERS=6
#   GTX 1080/2080 (8-11GB):  BATCH_SIZE=64,  NUM_WORKERS=4
#   CPU fallback:             BATCH_SIZE=32,  NUM_WORKERS=0

BATCH_SIZE   = 128   # ← adjust for your GPU
EPOCHS       = 60
LR           = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 4     # ← set to 0 on Windows if DataLoader errors occur

PROGRESS_FILE = os.path.join(CHECKPOINT_DIR, 'progress.json')

print(f'Device:    {DEVICE}')
print(f'Batch:     {BATCH_SIZE}')
print(f'Epochs:    {EPOCHS}')
print(f'Workers:   {NUM_WORKERS}')
print(f'Progress:  {PROGRESS_FILE}')

Device:    cuda
Batch:     128
Epochs:    60
Workers:   4
Progress:  ./checkpoints/progress.json


In [17]:
# ── Cell 7: Training helpers ─────────────────────────────────────
# !! Keep in sync with v1 Cell 7 and v3 Cell 8 — identical code !!
from sklearn.metrics import accuracy_score

def train_epoch(model, loader, opt, criterion, scaler=None):
    model.train()
    total_loss = correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                out = model(x); loss = criterion(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            out = model(x); loss = criterion(out, y)
            loss.backward(); opt.step()
        total_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        preds.extend(model(x.to(DEVICE)).argmax(1).cpu().tolist())
        labels.extend(y.tolist())
    return np.array(labels), np.array(preds)

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f: return json.load(f)
    return {'completed_folds': [], 'fold_results': []}

def save_progress(prog):
    with open(PROGRESS_FILE, 'w') as f: json.dump(prog, f, indent=2)

print('Training helpers defined.')

Training helpers defined.


In [18]:
# ── Cell 8: SVM Baseline ─────────────────────────────────────────
# Skips automatically if svm_results.json already exists
# !! Keep in sync with v1 Cell 8 and v3 Cell 9 — identical code !!
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

SVM_RESULTS_FILE = os.path.join(RESULTS_DIR, 'svm_results.json')

if os.path.exists(SVM_RESULTS_FILE):
    print('SVM results already exist. Loading...')
    with open(SVM_RESULTS_FILE) as f:
        svm_summary = json.load(f)
    print(f"SVM Mean: {svm_summary['summary']['mean_accuracy']*100:.2f}%")
else:
    meta = load_metadata(DATASET_ROOT)
    mfcc_cache = os.path.join(CACHE_DIR, 'mfcc.npz')
    if os.path.exists(mfcc_cache):
        data = np.load(mfcc_cache)
        X_all, y_all, folds_all = data['X'], data['y'], data['folds']
        print('Loaded MFCC cache.')
    else:
        X_all, y_all, folds_all = [], [], []
        for i, row in meta.iterrows():
            path = audio_path(DATASET_ROOT, row['fold'], row['slice_file_name'])
            X_all.append(extract_mfcc(load_audio(path)))
            y_all.append(row['classID'])
            folds_all.append(row['fold'])
            if (i+1) % 1000 == 0: print(f'  MFCC {i+1}/{len(meta)}')
        X_all = np.array(X_all, dtype=np.float32)
        y_all = np.array(y_all, dtype=np.int64)
        folds_all = np.array(folds_all)
        np.savez(mfcc_cache, X=X_all, y=y_all, folds=folds_all)
        print('MFCC cache saved.')

    svm_fold_results = []
    for fold, train_df, test_df in get_fold_splits(meta):
        clf = Pipeline([('sc', StandardScaler()),
                         ('svm', SVC(kernel='rbf', C=10, gamma='scale', cache_size=1000))])
        tm = folds_all != fold; te = folds_all == fold
        print(f'  SVM fold {fold}...')
        clf.fit(X_all[tm], y_all[tm])
        preds = clf.predict(X_all[te])
        acc = accuracy_score(y_all[te], preds)
        svm_fold_results.append({'fold': int(fold), 'accuracy': float(acc)})
        print(f'  Fold {fold}: {acc*100:.2f}%')

    accs = [r['accuracy'] for r in svm_fold_results]
    svm_summary = {
        'model': 'SVM+MFCC',
        'summary': {'mean_accuracy': float(np.mean(accs)), 'std_accuracy': float(np.std(accs))},
        'folds': svm_fold_results
    }
    with open(SVM_RESULTS_FILE, 'w') as f: json.dump(svm_summary, f, indent=2)
    print(f'\nSVM Mean: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%')

FileNotFoundError: [Errno 2] No such file or directory: './UrbanSound8K/metadata/UrbanSound8K.csv'

In [ ]:
# ── Cell 9: CNN Training (multi-session) ─────────────────────────
# !! Keep in sync with v1 Cell 9 and v3 Cell 10 — identical logic !!
# Checkpoint format: foldN_best.pt (weights only) + foldN_resume.pt (full state)
# progress.json format: {completed_folds: [int], fold_results: [{fold, accuracy}]}

meta = load_metadata(DATASET_ROOT)
prog = load_progress()
completed = set(prog['completed_folds'])
fold_results = prog['fold_results']

scaler = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

for test_fold, train_df, test_df in get_fold_splits(meta):
    if test_fold in completed:
        matching = [r for r in fold_results if r.get('fold') == test_fold]
        if matching:
            print(f'[SKIP] Fold {test_fold} — {matching[0]["accuracy"]*100:.2f}%')
        continue

    print(f'\n── Fold {test_fold} ── {len(train_df)} train / {len(test_df)} test')

    ckpt_best   = os.path.join(CHECKPOINT_DIR, f'fold{test_fold}_best.pt')
    ckpt_resume = os.path.join(CHECKPOINT_DIR, f'fold{test_fold}_resume.pt')

    pin = (DEVICE.type == 'cuda')
    train_ds = MelDataset(train_df, DATASET_ROOT, augment=True)
    test_ds  = MelDataset(test_df,  DATASET_ROOT, augment=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=pin,
                               persistent_workers=(NUM_WORKERS > 0))
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=pin)

    model = UrbanCNN().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit  = nn.CrossEntropyLoss()

    start_epoch = 1
    best_acc    = 0.0

    if os.path.exists(ckpt_resume):
        state = torch.load(ckpt_resume, map_location=DEVICE)
        model.load_state_dict(state['model'])
        opt.load_state_dict(state['optimizer'])
        sched.load_state_dict(state['scheduler'])
        start_epoch = state['epoch'] + 1
        best_acc    = state['best_acc']
        print(f'Resumed from epoch {start_epoch}, best_acc={best_acc*100:.2f}%')

    for epoch in range(start_epoch, EPOCHS + 1):
        loss, train_acc = train_epoch(model, train_loader, opt, crit, scaler)
        sched.step()

        if epoch % 5 == 0 or epoch == EPOCHS:
            torch.save({'epoch': epoch, 'model': model.state_dict(),
                         'optimizer': opt.state_dict(), 'scheduler': sched.state_dict(),
                         'best_acc': best_acc}, ckpt_resume)
            labels, preds = eval_model(model, test_loader)
            val_acc = (labels == preds).mean()
            print(f'  E{epoch:02d}/{EPOCHS}  loss={loss:.4f}  '
                  f'train={train_acc*100:.1f}%  val={val_acc*100:.1f}%')
            if val_acc > best_acc:
                best_acc = val_acc
                torch.save(model.state_dict(), ckpt_best)

    model.load_state_dict(torch.load(ckpt_best, map_location=DEVICE))
    labels, preds = eval_model(model, test_loader)
    acc = float((labels == preds).mean())

    fold_results.append({'fold': int(test_fold), 'accuracy': acc})
    completed.add(test_fold)
    prog['completed_folds'] = list(completed)
    prog['fold_results']    = fold_results
    save_progress(prog)

    if os.path.exists(ckpt_resume): os.remove(ckpt_resume)
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    print(f'  Fold {test_fold} DONE — {acc*100:.2f}%')

print('\n── All folds complete ──')

In [ ]:
# ── Cell 10: Final summary ────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

prog = load_progress()
cnn_accs = [r['accuracy'] for r in prog['fold_results']]
print('CNN Results:')
for r in sorted(prog['fold_results'], key=lambda x: x['fold']):
    print(f'  Fold {r["fold"]:2d}: {r["accuracy"]*100:.2f}%')
print(f'  Mean: {np.mean(cnn_accs)*100:.2f}% ± {np.std(cnn_accs)*100:.2f}%')

if os.path.exists(SVM_RESULTS_FILE):
    with open(SVM_RESULTS_FILE) as f: svm = json.load(f)
    svm_accs = [r['accuracy'] for r in svm['folds']]
    print(f"\nSVM Baseline: {np.mean(svm_accs)*100:.2f}% ± {np.std(svm_accs)*100:.2f}%")

cnn_summary = {
    'model': 'UrbanCNN',
    'summary': {'mean_accuracy': float(np.mean(cnn_accs)), 'std_accuracy': float(np.std(cnn_accs))},
    'folds': prog['fold_results']
}
with open(os.path.join(RESULTS_DIR, 'cnn_results.json'), 'w') as f:
    json.dump(cnn_summary, f, indent=2)
print('\nResults saved.')

In [ ]:
# ── Cell 11: Confusion matrix (run after all folds complete) ──────
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

meta = load_metadata(DATASET_ROOT)
all_labels, all_preds = [], []

for test_fold, train_df, test_df in get_fold_splits(meta):
    ckpt = os.path.join(CHECKPOINT_DIR, f'fold{test_fold}_best.pt')
    if not os.path.exists(ckpt):
        print(f'Missing checkpoint for fold {test_fold}'); continue
    test_ds = MelDataset(test_df, DATASET_ROOT, augment=False)
    loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    model   = UrbanCNN().to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    labels, preds = eval_model(model, loader)
    all_labels.extend(labels.tolist())
    all_preds.extend(preds.tolist())
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

print(f'Overall accuracy: {accuracy_score(all_labels, all_preds)*100:.2f}%')
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — UrbanCNN (all folds)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print('Saved.')

In [ ]:
# ── Cell 12: Git push ────────────────────────────────────────────
import subprocess

commit_msg = 'update v2 notebook'  # ← edit before running

subprocess.run(['git', 'config', '--global', 'user.email', 'dun011@ucsd.edu'])
subprocess.run(['git', 'config', '--global', 'user.name',  'DanielNg520'])
subprocess.run(['git', 'add', 'v2_cuda.ipynb'])
r = subprocess.run(['git', 'commit', '-m', commit_msg], capture_output=True, text=True)
print(r.stdout)
r = subprocess.run(['git', 'push'], capture_output=True, text=True)
print(r.stdout or r.stderr)